<a href="https://colab.research.google.com/github/sneharc16/CFI-Assignment-2-Final/blob/main/CFI_Assignment_2_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================
# 1. Install dependencies
# =========================
!pip install -q yt-dlp ultralytics opencv-python pandas numpy tqdm plotly

import os
import cv2
import json
import math
import torch
import shutil
import pandas as pd
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
from IPython.display import display

# =========================
# 2. Config
# =========================
YOUTUBE_URL = "https://www.youtube.com/watch?v=bqLVwMFqkdA"

BASE_DIR = "/content/cfi_assignment"
VIDEO_PATH = f"{BASE_DIR}/video.mp4"
FRAMES_DIR = f"{BASE_DIR}/frames"
ANNOT_DIR = f"{BASE_DIR}/annotated_top_frames"
OUTPUT_JSON = f"{BASE_DIR}/results.json"
DASHBOARD_HTML = f"{BASE_DIR}/dashboard.html"

SAMPLE_EVERY_SEC = 2.0       # 2 sec = ~1800 frames for 1 hour
IMGSZ = 640
CONF_THRES = 0.25

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(ANNOT_DIR, exist_ok=True)

# =========================
# 3. GPU / CUDA check
# =========================
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = "cuda"
else:
    print("WARNING: GPU not available. Runtime > Change runtime type > T4 GPU")
    device = "cpu"

!nvidia-smi

# =========================
# 4. Download YouTube video
# =========================
!yt-dlp -f "mp4[height<=720]/best[height<=720]/best" -o "{VIDEO_PATH}" "{YOUTUBE_URL}"

print("Downloaded video:", VIDEO_PATH)
print("Size MB:", os.path.getsize(VIDEO_PATH) / 1e6)

# =========================
# 5. Read video metadata
# =========================
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration_sec = total_frames / fps

print("FPS:", fps)
print("Total frames:", total_frames)
print("Duration sec:", duration_sec)
print("Duration min:", duration_sec / 60)

frame_step = max(1, int(round(fps * SAMPLE_EVERY_SEC)))
print("Processing every", frame_step, "frames")

# =========================
# 6. Load lightweight YOLO model
# =========================
# Try YOLO11n first; fallback to YOLOv8n if unavailable
try:
    model = YOLO("yolo11n.pt")
except Exception:
    model = YOLO("yolov8n.pt")

model.to(device)

# COCO classes used:
# 0 person, 1 bicycle, 2 car, 3 motorcycle, 5 bus, 7 truck, 9 traffic light, 67 cell phone
TARGET_CLASSES = [0, 1, 2, 3, 5, 7, 9, 67]

COCO_TO_CATEGORY = {
    1: "Others",       # bicycle
    2: "LMV",          # car
    3: "Two-Wheeler",  # motorcycle
    5: "HMV",          # bus
    7: "HMV",          # truck
}

def sec_to_hhmmss(sec):
    sec = int(round(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

def box_iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - inter + 1e-9
    return inter / union

def center_inside(box_small, box_big):
    x1, y1, x2, y2 = box_small
    bx1, by1, bx2, by2 = box_big
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    return bx1 <= cx <= bx2 and by1 <= cy <= by2

# =========================
# 7. Run tracking + lightweight violation heuristics
# =========================
unique_vehicles = {}
density_rows = []
event_log = []

top_violation_candidates = {
    "triple_riding": [],
    "mobile_phone_use": [],
    "signal_jumping_candidate": [],
    "helmetless_candidate": [],
    "wrong_side_candidate": []
}

prev_centers = {}

frame_idx = 0
processed = 0

pbar = tqdm(total=math.ceil(total_frames / frame_step))

while frame_idx < total_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        break

    timestamp_sec = frame_idx / fps
    timestamp = sec_to_hhmmss(timestamp_sec)

    # YOLO tracking
    results = model.track(
        source=frame,
        persist=True,
        conf=CONF_THRES,
        imgsz=IMGSZ,
        classes=TARGET_CLASSES,
        verbose=False,
        device=device
    )

    r = results[0]
    boxes = []
    if r.boxes is not None and len(r.boxes) > 0:
        xyxy = r.boxes.xyxy.detach().cpu().numpy()
        cls = r.boxes.cls.detach().cpu().numpy().astype(int)
        conf = r.boxes.conf.detach().cpu().numpy()

        if r.boxes.id is not None:
            track_ids = r.boxes.id.detach().cpu().numpy().astype(int)
        else:
            track_ids = np.array([-1] * len(cls))

        for b, c, cf, tid in zip(xyxy, cls, conf, track_ids):
            b = [float(x) for x in b]
            boxes.append({
                "bbox": b,
                "class_id": int(c),
                "class_name": model.names[int(c)],
                "confidence": float(cf),
                "track_id": int(tid)
            })

    # Vehicle count and density
    vehicle_boxes = [x for x in boxes if x["class_id"] in COCO_TO_CATEGORY]
    person_boxes = [x for x in boxes if x["class_id"] == 0]
    phone_boxes = [x for x in boxes if x["class_id"] == 67]
    traffic_light_boxes = [x for x in boxes if x["class_id"] == 9]
    motorcycle_boxes = [x for x in boxes if x["class_id"] == 3]

    density_rows.append({
        "timestamp": timestamp,
        "timestamp_sec": timestamp_sec,
        "vehicles_visible": len(vehicle_boxes)
    })

    for vb in vehicle_boxes:
        tid = vb["track_id"]
        if tid != -1 and tid not in unique_vehicles:
            unique_vehicles[tid] = {
                "first_seen": timestamp,
                "category": COCO_TO_CATEGORY[vb["class_id"]],
                "class_name": vb["class_name"],
                "confidence": vb["confidence"]
            }

    # Heuristic 1: Triple riding candidate
    for mb in motorcycle_boxes:
        riders = 0
        for pb in person_boxes:
            if center_inside(pb["bbox"], mb["bbox"]) or box_iou_xyxy(pb["bbox"], mb["bbox"]) > 0.05:
                riders += 1

        if riders >= 3:
            top_violation_candidates["triple_riding"].append({
                "timestamp": timestamp,
                "timestamp_sec": timestamp_sec,
                "confidence": min(0.99, 0.5 + 0.15 * riders),
                "bbox": mb["bbox"],
                "frame_idx": frame_idx,
                "reason": f"{riders} persons overlapping motorcycle"
            })

    # Heuristic 2: Mobile phone use candidate
    for ph in phone_boxes:
        for pb in person_boxes:
            if box_iou_xyxy(ph["bbox"], pb["bbox"]) > 0.01 or center_inside(ph["bbox"], pb["bbox"]):
                top_violation_candidates["mobile_phone_use"].append({
                    "timestamp": timestamp,
                    "timestamp_sec": timestamp_sec,
                    "confidence": ph["confidence"],
                    "bbox": ph["bbox"],
                    "frame_idx": frame_idx,
                    "reason": "cell phone detected close to person"
                })

    # Heuristic 3: Helmetless candidate
    # No helmet detector in standard COCO. So every visible 2W rider becomes a candidate for manual review.
    for mb in motorcycle_boxes:
        if len(person_boxes) > 0:
            top_violation_candidates["helmetless_candidate"].append({
                "timestamp": timestamp,
                "timestamp_sec": timestamp_sec,
                "confidence": mb["confidence"] * 0.5,
                "bbox": mb["bbox"],
                "frame_idx": frame_idx,
                "reason": "2W rider visible; helmet requires manual/zero-shot verification"
            })

    # Heuristic 4: Signal jumping candidate
    # Real signal jumping needs red-light state + stop-line crossing. Here we only flag frames with traffic light and nearby vehicles.
    if len(traffic_light_boxes) > 0 and len(vehicle_boxes) > 0:
        best_v = max(vehicle_boxes, key=lambda x: x["confidence"])
        top_violation_candidates["signal_jumping_candidate"].append({
            "timestamp": timestamp,
            "timestamp_sec": timestamp_sec,
            "confidence": max([x["confidence"] for x in traffic_light_boxes]),
            "bbox": best_v["bbox"],
            "frame_idx": frame_idx,
            "reason": "traffic light + vehicle present; red/stopline needs manual verification"
        })

    # Heuristic 5: Wrong-side candidate
    # Approximation: track motion direction opposite to majority motion.
    current_centers = {}
    motions = []
    for vb in vehicle_boxes:
        tid = vb["track_id"]
        if tid == -1:
            continue
        x1, y1, x2, y2 = vb["bbox"]
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        current_centers[tid] = (cx, cy, vb)

        if tid in prev_centers:
            px, py, _ = prev_centers[tid]
            dy = cy - py
            motions.append((tid, dy, vb))

    if len(motions) >= 4:
        median_dy = np.median([m[1] for m in motions])
        for tid, dy, vb in motions:
            if abs(dy) > 10 and np.sign(dy) != np.sign(median_dy):
                top_violation_candidates["wrong_side_candidate"].append({
                    "timestamp": timestamp,
                    "timestamp_sec": timestamp_sec,
                    "confidence": vb["confidence"] * 0.6,
                    "bbox": vb["bbox"],
                    "frame_idx": frame_idx,
                    "reason": "vehicle motion opposite to majority tracked flow"
                })

    prev_centers = current_centers

    processed += 1
    frame_idx += frame_step
    pbar.update(1)

pbar.close()
cap.release()

print("Processed frames:", processed)
print("Unique tracked vehicles:", len(unique_vehicles))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Torch version: 2.10.0+cpu
CUDA available: False
/bin/bash: line 1: nvidia-smi: command not found
[youtube] Extracting URL: https://www.youtube.com/watch?v=bqLVwMFqkdA
[youtube] bqLVwMFqkdA: Downloading webpage
[youtube] bqLVwMFqkdA: Downloading android vr player API JSON
[info] bqLVwMFqkdA: Downloading 1 format(s): 18
[download] Destination: /content/cfi_assignment/video.mp4
[download] 100% of  303.97MiB in 00:00:18 at 16.47MiB/s
Downloaded video: /c

  0%|          | 0/2097 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 332ms
Prepared 1 package in 75ms
Installed 1 package in 4ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|█████████▉| 2096/2097 [11:24<00:00,  4.40it/s]

WARNING ⚠️ not enough matching points


100%|██████████| 2097/2097 [11:24<00:00,  3.06it/s]

Processed frames: 2097
Unique tracked vehicles: 432


In [2]:
# =========================
# 8. Save top 3 annotated frames per violation type
# =========================

def draw_box_and_label(frame, bbox, label):
    x1, y1, x2, y2 = map(int, bbox)
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
    cv2.putText(
        frame,
        label,
        (x1, max(30, y1 - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2
    )
    return frame

top3_outputs = {}

cap = cv2.VideoCapture(VIDEO_PATH)

for violation_type, candidates in top_violation_candidates.items():
    candidates = sorted(candidates, key=lambda x: x["confidence"], reverse=True)
    top3 = candidates[:3]
    saved = []

    for i, item in enumerate(top3, start=1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, item["frame_idx"])
        ok, frame = cap.read()
        if not ok:
            continue

        label = f"{violation_type} {item['confidence']:.2f} {item['timestamp']}"
        frame = draw_box_and_label(frame, item["bbox"], label)

        out_path = f"{ANNOT_DIR}/{violation_type}_top{i}_{item['timestamp'].replace(':','-')}.jpg"
        cv2.imwrite(out_path, frame)

        saved.append({
            "timestamp": item["timestamp"],
            "confidence": item["confidence"],
            "bbox": item["bbox"],
            "image_path": out_path,
            "reason": item["reason"]
        })

    top3_outputs[violation_type] = saved

cap.release()

print("Annotated frames saved to:", ANNOT_DIR)

Annotated frames saved to: /content/cfi_assignment/annotated_top_frames


In [3]:
# =========================
# 9. Create junction manual review template
# =========================

JUNCTION_REVIEW_DIR = f"{BASE_DIR}/junction_review_frames"
os.makedirs(JUNCTION_REVIEW_DIR, exist_ok=True)

JUNCTION_SAMPLE_EVERY_SEC = 10
junction_step = int(round(fps * JUNCTION_SAMPLE_EVERY_SEC))

cap = cv2.VideoCapture(VIDEO_PATH)

junction_rows = []
frame_idx = 0

while frame_idx < total_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        break

    timestamp_sec = frame_idx / fps
    timestamp = sec_to_hhmmss(timestamp_sec)

    out_path = f"{JUNCTION_REVIEW_DIR}/junction_candidate_{timestamp.replace(':','-')}.jpg"
    cv2.imwrite(out_path, frame)

    junction_rows.append({
        "timestamp": timestamp,
        "timestamp_sec": timestamp_sec,
        "frame_path": out_path,
        "is_junction": 0,
        "junction_type": "",
        "notes": ""
    })

    frame_idx += junction_step

cap.release()

junction_template = f"{BASE_DIR}/junction_manual_review.csv"
pd.DataFrame(junction_rows).to_csv(junction_template, index=False)

print("Fill this CSV manually for junctions:", junction_template)

Fill this CSV manually for junctions: /content/cfi_assignment/junction_manual_review.csv


In [4]:
# =========================
# 10. Build final JSON
# =========================

vehicle_df = pd.DataFrame(unique_vehicles.values())

if len(vehicle_df) > 0:
    category_counts = vehicle_df["category"].value_counts().to_dict()
else:
    category_counts = {}

total_unique = int(len(unique_vehicles))

vehicle_category_summary = {}
for cat, cnt in category_counts.items():
    vehicle_category_summary[cat] = {
        "count": int(cnt),
        "percentage": float(100 * cnt / max(1, total_unique))
    }

density_df = pd.DataFrame(density_rows)
density_csv = f"{BASE_DIR}/vehicle_density.csv"
density_df.to_csv(density_csv, index=False)

# Read manual junction CSV if filled, else empty
junction_csv = f"{BASE_DIR}/junction_manual_review.csv"
if os.path.exists(junction_csv):
    jdf = pd.read_csv(junction_csv)
    if "is_junction" in jdf.columns:
        jdf = jdf[jdf["is_junction"] == 1].copy()
    else:
        jdf = pd.DataFrame()
else:
    jdf = pd.DataFrame()

junction_events = []
if len(jdf) > 0:
    for _, row in jdf.iterrows():
        junction_events.append({
            "timestamp": row.get("timestamp", ""),
            "timestamp_sec": float(row.get("timestamp_sec", 0)),
            "type": row.get("junction_type", "Unknown"),
            "frame_path": row.get("frame_path", "")
        })

junction_type_counts = {}
for x in junction_events:
    junction_type_counts[x["type"]] = junction_type_counts.get(x["type"], 0) + 1

violations = {}
for vtype, candidates in top_violation_candidates.items():
    # Keep candidate timestamps de-duplicated roughly by timestamp
    seen = set()
    events = []
    for item in sorted(candidates, key=lambda x: x["timestamp_sec"]):
        bucket = int(item["timestamp_sec"] // 5) * 5
        key = (vtype, bucket)
        if key in seen:
            continue
        seen.add(key)

        events.append({
            "timestamp": item["timestamp"],
            "timestamp_sec": item["timestamp_sec"],
            "confidence": item["confidence"],
            "bbox": item["bbox"],
            "reason": item["reason"]
        })

    violations[vtype] = {
        "count": len(events),
        "events": events,
        "top_3_annotated": top3_outputs.get(vtype, [])
    }

results = {
    "metadata": {
        "video_url": YOUTUBE_URL,
        "video_path": VIDEO_PATH,
        "sample_every_sec": SAMPLE_EVERY_SEC,
        "fps": fps,
        "duration_sec": duration_sec,
        "model": "YOLO lightweight pretrained model",
        "device": device,
        "limitations": [
            "No task-specific annotations were available.",
            "Vehicle counts are based on pretrained object detection and tracking.",
            "Helmetless, signal jumping, wrong-side driving, and phone use are candidate detections requiring manual verification.",
            "Junction type classification is produced through sampled-frame manual review."
        ]
    },
    "summary": {
        "total_violations_candidates": int(sum(v["count"] for v in violations.values())),
        "total_junctions": int(len(junction_events)),
        "total_unique_vehicles": total_unique
    },
    "violations": violations,
    "junctions": {
        "total_count": int(len(junction_events)),
        "type_wise_breakdown": junction_type_counts,
        "events": junction_events
    },
    "vehicles": {
        "total_unique_count": total_unique,
        "category_wise_count_percentage": vehicle_category_summary,
        "density_csv": density_csv,
        "density_over_time": density_rows[:2000]
    }
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(results, f, indent=2)

print("Saved JSON:", OUTPUT_JSON)

Saved JSON: /content/cfi_assignment/results.json


In [5]:
# =========================
# 11. Build simple HTML dashboard
# =========================

with open(OUTPUT_JSON, "r") as f:
    results = json.load(f)

viol_counts = {
    k: v["count"]
    for k, v in results["violations"].items()
}

veh_counts = {
    k: v["count"]
    for k, v in results["vehicles"]["category_wise_count_percentage"].items()
}

events_flat = []

for vtype, obj in results["violations"].items():
    for e in obj["events"]:
        events_flat.append({
            "timestamp": e["timestamp"],
            "timestamp_sec": e["timestamp_sec"],
            "event_type": vtype,
            "details": e.get("reason", "")
        })

for e in results["junctions"]["events"]:
    events_flat.append({
        "timestamp": e["timestamp"],
        "timestamp_sec": e["timestamp_sec"],
        "event_type": "junction_" + e["type"],
        "details": e.get("type", "")
    })

events_flat = sorted(events_flat, key=lambda x: x["timestamp_sec"])

html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Crashfree India Dashcam Analysis Dashboard</title>
    <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 24px;
            background: #f7f7f7;
        }}
        .cards {{
            display: flex;
            gap: 16px;
            margin-bottom: 24px;
        }}
        .card {{
            background: white;
            padding: 20px;
            border-radius: 12px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.08);
            flex: 1;
        }}
        .card h2 {{
            margin: 0;
            font-size: 32px;
        }}
        .grid {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 24px;
        }}
        .panel {{
            background: white;
            padding: 20px;
            border-radius: 12px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.08);
            margin-bottom: 24px;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            background: white;
        }}
        th, td {{
            padding: 10px;
            border-bottom: 1px solid #ddd;
            text-align: left;
        }}
        tr:hover {{
            background: #f0f0f0;
            cursor: pointer;
        }}
        video {{
            width: 100%;
            max-height: 480px;
            background: black;
            border-radius: 12px;
        }}
    </style>
</head>
<body>

<h1>Crashfree India Dashcam Analysis Dashboard</h1>

<div class="panel">
    <video id="dashcamVideo" controls>
        <source src="video.mp4" type="video/mp4">
    </video>
</div>

<div class="cards">
    <div class="card">
        <p>Total Violation Candidates</p>
        <h2>{results["summary"]["total_violations_candidates"]}</h2>
    </div>
    <div class="card">
        <p>Total Junctions</p>
        <h2>{results["summary"]["total_junctions"]}</h2>
    </div>
    <div class="card">
        <p>Total Unique Vehicles</p>
        <h2>{results["summary"]["total_unique_vehicles"]}</h2>
    </div>
</div>

<div class="grid">
    <div class="panel">
        <h2>Violation Breakdown</h2>
        <div id="violChart"></div>
    </div>
    <div class="panel">
        <h2>Vehicle Category Distribution</h2>
        <div id="vehChart"></div>
    </div>
</div>

<div class="panel">
    <h2>Event Log</h2>
    <table>
        <thead>
            <tr>
                <th>Timestamp</th>
                <th>Event Type</th>
                <th>Details</th>
            </tr>
        </thead>
        <tbody>
"""

for e in events_flat:
    html += f"""
            <tr onclick="seekVideo({float(e['timestamp_sec'])})">
                <td>{e['timestamp']}</td>
                <td>{e['event_type']}</td>
                <td>{e['details']}</td>
            </tr>
    """

html += f"""
        </tbody>
    </table>
</div>

<script>
function seekVideo(sec) {{
    const video = document.getElementById("dashcamVideo");
    video.currentTime = sec;
    video.play();
}}

const violCounts = {json.dumps(viol_counts)};
const vehCounts = {json.dumps(veh_counts)};

Plotly.newPlot("violChart", [{{
    x: Object.keys(violCounts),
    y: Object.values(violCounts),
    type: "bar"
}}], {{
    margin: {{ t: 20 }}
}});

Plotly.newPlot("vehChart", [{{
    labels: Object.keys(vehCounts),
    values: Object.values(vehCounts),
    type: "pie"
}}], {{
    margin: {{ t: 20 }}
}});
</script>

</body>
</html>
"""

with open(DASHBOARD_HTML, "w") as f:
    f.write(html)

print("Dashboard saved:", DASHBOARD_HTML)
with open(DASHBOARD_HTML, "w") as f:
    f.write(html)

# Video is already inside BASE_DIR as video.mp4, so no need to copy it again
print("Dashboard saved:", DASHBOARD_HTML)
print("Video path:", VIDEO_PATH)

Dashboard saved: /content/cfi_assignment/dashboard.html
Dashboard saved: /content/cfi_assignment/dashboard.html
Video path: /content/cfi_assignment/video.mp4


In [6]:
zip_path = "/content/cfi_assignment_output.zip"
shutil.make_archive("/content/cfi_assignment_output", "zip", BASE_DIR)

print("Download this zip:", zip_path)

Download this zip: /content/cfi_assignment_output.zip


In [7]:
from google.colab import files

files.download("/content/cfi_assignment_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>